# पाठ ११ - मोडेल सन्दर्भ प्रोटोकल (MCP)

**मोडेल सन्दर्भ प्रोटोकल (MCP)** एउटा खुला मापदण्ड हो जसले एजेन्टहरूलाई रनटाइममा उपकरणहरू, स्रोतहरू, र डेटा स्रोतहरू गतिशील रूपमा पत्ता लगाउन र प्रयोग गर्न सक्षम बनाउँछ। एजेन्टमा उपकरणहरू हार्डकोड गर्ने सट्टा, MCP ले एजेन्टहरूलाई आवश्यकताअनुसार क्षमताहरू प्रदर्शन गर्ने बाह्य सर्भरहरूमा जडान हुन दिन्छ।

यस पाठमा, तपाईं सिक्नुहुनेछ:
- MCP के हो र किन यो एजेन्ट प्रणालीहरूका लागि महत्वपूर्ण छ
- MCP क्लाएन्ट-सर्भर संरचना कसरी काम गर्छ
- MCP-शैली उपकरण पत्ता लगाउने प्रयोग गर्ने एजेन्टहरू कसरी बनाउने


## सेटअप

**पूर्व शर्तहरू:**
- एक Microsoft Foundry परियोजना जुन मोडेलसँग तैनाथ गरिएको छ
- प्रमाणिकरणको लागि `az login` चलाउनुहोस्


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## मोडेल सन्दर्भ प्रोटोकल (MCP) के हो?

MCP ले AI एजेन्टहरूले बाह्य उपकरण र डेटा स्रोतहरू पत्ता लगाउने र अन्तरक्रिया गर्ने एक मानक तरिका निर्धारण गर्दछ:

- **MCP सर्भर**: उपकरणहरू, स्रोतहरू, र प्रम्प्टहरू मानक प्रोटोकल मार्फत प्रदान गर्दछ
- **MCP क्लाएन्ट**: एजेन्ट रनटाइम जसले सर्भरसँग जडान गरी उपलब्ध क्षमता पत्ता लगाउँछ
- **गतिशील खोज**: एजेन्टहरूले हार्डकोड गरिएको उपकरण आवश्यक पर्दैन — उनीहरूले रनटाइममा उपलब्ध कुरा पत्ता लगाउँछन्

यसले विस्तारयोग्य एजेन्ट प्रणालीहरू निर्माण गर्न शक्तिशाली बनाउँछ जहाँ नयाँ क्षमता थप्न एजेन्ट कोडमा परिवर्तन नगरी गर्न सकिन्छ।


## कसरी MCP काम गर्छ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. एजेन्ट (MCP क्लाइन्ट) ले MCP सर्भरसँग जडान गर्छ
2. सर्भरले उपलब्ध उपकरणहरूको सूची र तिनीहरूको स्किमासँग प्रतिक्रिया दिन्छ
3. एजेन्टले त्यसपछि आफ्नो तर्कमा कुनै पनि फेला पारिएको उपकरण बोलाउन सक्छ
4. नतिजा त्यहि प्रोटोकल मार्फत फर्कन्छ


## MCP उपकरण पत्ता लगाउने अनुकरण गर्दै

किनभने वास्तविक MCP सर्भरले चलिरहेको सर्भर प्रक्रिया आवश्यक पर्दछ, हामी `@tool` कार्यहरू प्रयोग गरेर नमुना प्रस्तुत गर्नेछौं जसले MCP-सम्पर्कित आवास सेवा प्रदान गर्ने कुरा नक्कल गर्छ।

उत्पादनमा, यी उपकरणहरू स्थानीय रूपमा परिभाषित नभई MCP सर्भरबाट गतिशील रूपमा पत्ता लगाइने थिए।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## एमसीपी-शैली उपकरणहरूसँग एजेन्ट निर्माण गर्दै


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## उत्पादनमा MCP

उत्पादन वातावरणमा, MCP ले शक्तिशाली ढाँचाहरू सक्षम बनाउँछ:

- **डायनेमिक उपकरण पत्ता लगाउने**: एजेन्टहरू MCP सर्भरहरूमा जडान हुन्छन् र रनटाइममा उपकरणहरू पत्ता लगाउँछन्
- **अलग ढांचा**: उपकरण प्रदायकहरूले एजेन्टबाट स्वतन्त्र रूपमा अद्यावधिक गर्न सक्छन्
- **क्रमिक संगठन साझेदारी**: टोलीहरूले MCP सर्भरहरू मार्फत क्षमता एक्स्पोज गर्न सक्छन् जुन कुनै पनि एजेन्टले प्रयोग गर्न सक्छ
- **माइक्रोसफ्ट एजेन्ट फ्रेमवर्क समर्थन**: MAF ले `mcp` एकीकरण मार्फत एमसीपी क्लाइन्ट समर्थन समावेश गर्दछ

MAF सँग वास्तविक MCP सर्भर प्रयोग गर्नको लागि, तपाईं `hosted_mcp_tool()` वा MCP क्लाइन्ट एकीकरण मार्फत जडान गर्नुहुनेछ।

**थप जान्न:**
- [MCP विनिर्देश](https://modelcontextprotocol.io/)
- [माइक्रोसफ्ट एजेन्ट फ्रेमवर्क MCP समर्थन](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## सारांश

यस पाठमा, तपाईंले सिक्नुभयो:
- **MCP** एक खुला मानक हो जुन एजेन्टहरू र उपकरण प्रदायकहरू बीच गतिशील उपकरण पत्ता लगाउनको लागि हो
- **क्लाइन्ट-सर्भर वास्तुकला** ले एजेन्टहरूलाई रनटाइममा क्षमता पत्ता लगाउन अनुमति दिन्छ
- MCP ले **विस्तारयोग्य, पृथक एजेन्ट प्रणालीहरू** सक्षम पार्दछ जहाँ उपकरणहरू कोड परिवर्तन बिना थप्न सकिन्छ
- Microsoft एजेन्ट फ्रेमवर्कले उत्पादन प्रयोगको लागि **अन्तर्निर्मित MCP समर्थन** प्रदान गर्दछ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
